# Multiverse of Monsoon — GRAST v3 GPU Training (Anomaly + Damping + GroupNorm + EMA)

Self-contained Kaggle GPU training notebook implementing the v3 diagnosis fixes:
- **Anomaly formulation** using train-only 31-day smoothed daily climatology `clim[366, 6, 32, 36]` and per-channel `anom_std[6]` (eliminates the 7× `tp`-vs-`t2m` loss imbalance).
- **Learned per-channel damping** `a = sigmoid(damp_logit)` toward climatology (prevents late-lead double-penalty drift).
- **`GroupNorm` + EMA weights (`0.999`) + input noise (`0.03`)** for stable 10-day rollouts and causal shocks.
- **Straight-through physical clamps** for both `tp >= 0` and `tcwv >= 0`.
- **GPU-resident dataset** + `bf16` autocast (when supported) + early stopping + built-in Stage-1 benchmark.

> **Note:** Does not modify `kaggle_train.ipynb` or `src/`. Saves `/kaggle/working/artifacts/primary_grast_v3.pt`.

In [ ]:
import copy
import json
import math
import time
import zlib
from pathlib import Path
from typing import Tuple

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import uniform_filter1d

# ===========================================================================
# Configuration
# ===========================================================================
DATA_DIR_OVERRIDE = None
OUT_DIR = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_CH     = 64
HORIZON     = 8
BATCH_SIZE  = 64
DROPOUT     = 0.05
INPUT_NOISE = 0.03
EMA_DECAY   = 0.999
PATIENCE    = 8
SEED        = 42

# (horizon, max_epochs, lr)
CURRICULUM_PHASES = [
    (1, 40, 5.0e-4),
    (2, 30, 3.0e-4),
    (4, 30, 2.0e-4),
    (8, 30, 1.0e-4),
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = (device.type == "cuda") and torch.cuda.is_bf16_supported()
print(f"Device: {device} | bf16: {USE_BF16}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def find_data_files() -> Tuple[Path, Path]:
    if DATA_DIR_OVERRIDE is not None:
        d = Path(DATA_DIR_OVERRIDE)
        return d / "india_climate_train.nc", d / "stats.json"
    for root in [Path("/kaggle/input"), Path("data"), Path(".")]:
        if root.exists():
            nc_hits = list(root.rglob("india_climate_train.nc"))
            if nc_hits:
                nc_p = nc_hits[0]
                st_p = nc_p.parent / "stats.json"
                if st_p.exists():
                    return nc_p, st_p
    raise FileNotFoundError("Could not find india_climate_train.nc and stats.json.")

NC_PATH, STATS_PATH = find_data_files()
LOG_PATH = OUT_DIR / "training_log_v3.jsonl"
LOG_PATH.write_text("")  # fresh log per run
print(f"NC_PATH: {NC_PATH} | OUT_DIR: {OUT_DIR}")


In [ ]:
# ===========================================================================
# Data Loading, Train-Only Smoothed Climatology, & Geo Features
# ===========================================================================
CHANNELS = ["t2m", "tp", "u10", "v10", "msl", "tcwv"]
N_CHANNELS = 6
GRID_H, GRID_W = 32, 36
N_GEO_CHANNELS = 6
N_HISTORY = 3
N_DOY = 2
# 3*6 (anomaly history) + 6 (target-day norm clim) + 6 (geo) + 2 (doy_sincos) = 32
C_IN = N_HISTORY * N_CHANNELS + N_CHANNELS + N_GEO_CHANNELS + N_DOY

SPLIT = {
    "train": ("2005-01-01", "2017-12-31"),
    "val":   ("2018-01-01", "2019-12-31"),
    "test":  ("2020-01-01", "2021-12-31"),
}


def _read_dataset_all_chunks(h5file: h5py.File, name: str) -> np.ndarray:
    ds = h5file[name]
    plist = ds.id.get_create_plist()
    if plist.get_nfilters() == 0:
        return ds[()]
    has_deflate = any(plist.get_filter(i)[0] == h5py.h5z.FILTER_DEFLATE for i in range(plist.get_nfilters()))
    has_shuffle = any(plist.get_filter(i)[0] == h5py.h5z.FILTER_SHUFFLE for i in range(plist.get_nfilters()))
    if not has_deflate:
        return ds[()]
    out = np.empty(ds.shape, dtype=ds.dtype)
    elem_size = ds.dtype.itemsize
    with open(str(h5file.filename), "rb") as raw_fp:
        for ci in range(ds.id.get_num_chunks()):
            info = ds.id.get_chunk_info(ci)
            coff = info.chunk_offset
            raw_fp.seek(info.byte_offset)
            raw_bytes = np.frombuffer(zlib.decompress(raw_fp.read(info.size)), dtype=np.uint8)
            valid_shape = tuple(min(cs, s - co) for cs, s, co in zip(ds.chunks, ds.shape, coff))
            full_shape = tuple(ds.chunks)
            n_full = int(np.prod(full_shape))
            if has_shuffle:
                unshuffled = np.empty(n_full * elem_size, dtype=np.uint8)
                for b in range(elem_size):
                    unshuffled[b::elem_size] = raw_bytes[b * n_full : (b + 1) * n_full]
            else:
                unshuffled = raw_bytes
            chunk_arr = np.frombuffer(unshuffled.tobytes(), dtype=ds.dtype).reshape(full_shape)
            chunk_arr = chunk_arr[tuple(slice(0, vs) for vs in valid_shape)]
            out[tuple(slice(co, co + vs) for co, vs in zip(coff, valid_shape))] = chunk_arr
    return out


class DatasetReader:
    def __init__(self, nc_path: Path = NC_PATH) -> None:
        import xarray as xr
        try:
            ds = xr.open_dataset(nc_path, engine="h5netcdf")
            raw = ds["data"].values.astype("float32")
            static = ds["static"].values.astype("float32")
            lats = ds["lat"].values.astype("float32")
            lons = ds["lon"].values.astype("float32")
            dates = ds["time"].values.astype("datetime64[D]")
            ds.close()
        except OSError:
            with h5py.File(nc_path, "r") as f:
                raw = _read_dataset_all_chunks(f, "data").astype("float32")
                static = _read_dataset_all_chunks(f, "static").astype("float32")
                lats = _read_dataset_all_chunks(f, "lat").astype("float32")
                lons = _read_dataset_all_chunks(f, "lon").astype("float32")
            ds_meta = xr.open_dataset(nc_path, engine="h5netcdf")
            dates = ds_meta["time"].values.astype("datetime64[D]")
            ds_meta.close()

        self.data = raw
        self.static = static
        self.lats = lats
        self.lons = lons
        self._dates = dates
        yr_starts = dates.astype("datetime64[Y]").astype("datetime64[D]")
        self.doys = np.clip((dates - yr_starts).astype(int), 0, 365)  # 0..365

    def split_indices(self, split: str) -> np.ndarray:
        start, end = SPLIT[split]
        s, e = np.datetime64(start, "D"), np.datetime64(end, "D")
        return np.where((self._dates >= s) & (self._dates <= e))[0]

    def doy_sincos(self, day_idx: int) -> np.ndarray:
        doy = int(self.doys[day_idx]) + 1
        angle = 2.0 * np.pi * doy / 365.0
        return np.array([np.sin(angle), np.cos(angle)], dtype="float32")


def build_smoothed_climatology(reader: DatasetReader) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Train-only (2005-2017) 31-day smoothed daily climatology & anomaly std."""
    tr_idx = reader.split_indices("train")
    tr_doys = reader.doys[tr_idx]
    raw_clim = np.zeros((366, N_CHANNELS, GRID_H, GRID_W), dtype="float64")
    counts = np.zeros(366, dtype="int32")
    for idx, d in zip(tr_idx, tr_doys):
        raw_clim[d] += reader.data[idx]
        counts[d] += 1
    for d in range(366):
        raw_clim[d] = raw_clim[d] / counts[d] if counts[d] > 0 else raw_clim[d - 1]

    # Smooth across day-of-year (first 365 days circularly; copy 364 -> 365)
    smooth_365 = uniform_filter1d(raw_clim[:365], size=31, axis=0, mode="wrap")
    clim = np.empty((366, N_CHANNELS, GRID_H, GRID_W), dtype="float32")
    clim[:365] = smooth_365.astype("float32")
    clim[365] = clim[364]

    tr_anom = reader.data[tr_idx] - clim[tr_doys]
    anom_std = tr_anom.std(axis=(0, 2, 3)).astype("float32")
    anom_mean = clim.mean(axis=(0, 2, 3)).astype("float32")
    return clim, anom_std, anom_mean


def build_geo_features(lsm: np.ndarray, orography: np.ndarray, lats: np.ndarray, lons: np.ndarray) -> np.ndarray:
    H, W = lsm.shape
    lsm_ch = lsm.astype("float32")
    oro_min, oro_max = orography.min(), max(orography.max(), 1.0)
    oro_ch = ((orography - oro_min) / (oro_max - oro_min)).astype("float32")
    lat_ch = np.broadcast_to(((lats - 6.5) / (37.5 - 6.5) * 2 - 1).astype("float32")[:, None], (H, W)).copy()
    lon_ch = np.broadcast_to(((lons - 65.5) / (100.5 - 65.5) * 2 - 1).astype("float32")[None, :], (H, W)).copy()
    cos_lat_ch = np.broadcast_to(np.cos(np.deg2rad(lats)).astype("float32")[:, None], (H, W)).copy()
    dx_km = (6371.0 * np.cos(np.deg2rad(lats)) * np.deg2rad(1.0)).astype("float32")
    dx_ch = np.broadcast_to((dx_km / dx_km.max())[:, None], (H, W)).copy()
    return np.stack([lsm_ch, oro_ch, lat_ch, lon_ch, cos_lat_ch, dx_ch], axis=0)


In [ ]:
# ===========================================================================
# GRAST v3 Architecture (GroupNorm + Learned Damping + Anomaly Conditioning)
# ===========================================================================
class NonPeriodicPad(nn.Module):
    def __init__(self, pad: int = 1) -> None:
        super().__init__()
        self.pad = pad

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.pad(x, (self.pad,) * 4, mode="replicate")


def _gn(ch: int) -> nn.GroupNorm:
    return nn.GroupNorm(min(8, ch), ch)


class ResBlock(nn.Module):
    def __init__(self, channels: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.pad = NonPeriodicPad(1)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=0, bias=False)
        self.gn1   = _gn(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=0, bias=False)
        self.gn2   = _gn(channels)
        self.drop  = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.silu(self.gn1(self.conv1(self.pad(x))))
        h = self.drop(h)
        h = self.gn2(self.conv2(self.pad(h)))
        return F.silu(h + x)


class DownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.pad  = NonPeriodicPad(1)
        self.conv = nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=0, bias=False)
        self.gn   = _gn(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.silu(self.gn(self.conv(self.pad(x))))


class UpBlock(nn.Module):
    def __init__(self, coarse_ch: int, fine_ch: int, out_ch: int) -> None:
        super().__init__()
        self.pad  = NonPeriodicPad(1)
        self.conv = nn.Conv2d(coarse_ch + fine_ch, out_ch, 3, padding=0, bias=False)
        self.gn   = _gn(out_ch)

    def forward(self, coarse: torch.Tensor, fine: torch.Tensor) -> torch.Tensor:
        up = F.interpolate(coarse, size=fine.shape[-2:], mode="bilinear", align_corners=False)
        return F.silu(self.gn(self.conv(self.pad(torch.cat([up, fine], dim=1)))))


class GRASTv3(nn.Module):
    def __init__(self, c_in: int = C_IN, base_ch: int = 64, n_fine_pre: int = 2,
                 n_coarse: int = 3, n_fine_post: int = 2, dropout: float = 0.05) -> None:
        super().__init__()
        coarse_ch = base_ch * 2
        self.encoder = nn.Sequential(
            NonPeriodicPad(1),
            nn.Conv2d(c_in, base_ch, 3, padding=0, bias=False),
            _gn(base_ch),
            nn.SiLU(),
        )
        self.fine_pre  = nn.Sequential(*[ResBlock(base_ch, dropout) for _ in range(n_fine_pre)])
        self.down      = DownBlock(base_ch, coarse_ch)
        self.coarse    = nn.Sequential(*[ResBlock(coarse_ch, dropout) for _ in range(n_coarse)])
        self.up        = UpBlock(coarse_ch, base_ch, base_ch)
        self.fine_post = nn.Sequential(*[ResBlock(base_ch, dropout) for _ in range(n_fine_post)])
        self.head = nn.Sequential(
            NonPeriodicPad(1),
            nn.Conv2d(base_ch, base_ch // 2, 3, padding=0, bias=True),
            nn.SiLU(),
            NonPeriodicPad(1),
            nn.Conv2d(base_ch // 2, N_CHANNELS, 3, padding=0, bias=True),
        )
        self.residual_scale = nn.Parameter(torch.ones(N_CHANNELS) * 0.1)
        # sigmoid(2.44) ≈ 0.92 per-channel anomaly persistence damping
        self.damp_logit = nn.Parameter(torch.ones(N_CHANNELS) * 2.44)
        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        enc    = self.encoder(x)
        fine0  = self.fine_pre(enc)
        coarse = self.coarse(self.down(fine0))
        merged = self.up(coarse, fine0)
        feat   = self.fine_post(merged)
        dz     = self.head(feat)
        return dz * self.residual_scale.view(1, N_CHANNELS, 1, 1)


In [ ]:
# ===========================================================================
# Anomaly Rollout Engine (Straight-Through tp>=0 & tcwv>=0) + Loss
# ===========================================================================
RAIN_CH = 1
TCWV_CH = 5

def doy_idx_from_sincos(sin_v: torch.Tensor, cos_v: torch.Tensor) -> torch.Tensor:
    """Recover 0-based DOY index [0..364] from (sin, cos) of 2*pi*(doy+1)/365."""
    angle = torch.atan2(sin_v, cos_v) % (2.0 * math.pi)
    doy_1based = torch.round(angle * 365.0 / (2.0 * math.pi)).long()
    return ((doy_1based - 1) % 365).clamp(0, 364)


def anomaly_rollout(
    model: GRASTv3,
    geo_t: torch.Tensor,          # [1, 6, 32, 36]
    clim_t: torch.Tensor,         # [366, 6, 32, 36] physical
    clim_norm_t: torch.Tensor,    # [366, 6, 32, 36] centered/scaled
    anom_std_t: torch.Tensor,     # [6]
    anom_hist_t: torch.Tensor,    # [B, 3, 6, 32, 36] anomaly space
    doy_sin: torch.Tensor,        # [B] for last input state
    doy_cos: torch.Tensor,        # [B]
    steps: int,
    input_noise_std: float = 0.0,
) -> list:
    B, _, _, H, W = anom_hist_t.shape
    dev = anom_hist_t.device
    delta = 2.0 * math.pi / 365.0
    cd = math.cos(delta)
    sd = math.sin(delta)

    history = anom_hist_t.clone()
    if input_noise_std > 0.0:
        history = history + torch.randn_like(history) * input_noise_std

    sin_v, cos_v = doy_sin.clone(), doy_cos.clone()
    a = torch.sigmoid(model.damp_logit).view(1, N_CHANNELS, 1, 1)
    preds_anom = []

    for _ in range(steps):
        # advance DOY to target day
        next_sin = sin_v * cd + cos_v * sd
        next_cos = cos_v * cd - sin_v * sd
        tgt_doy = doy_idx_from_sincos(next_sin, next_cos)

        doy_map = torch.stack([sin_v, cos_v], dim=1)[:, :, None, None].expand(B, 2, H, W)
        geo_b = geo_t.expand(B, -1, -1, -1)
        clim_in = clim_norm_t[tgt_doy]  # [B, 6, H, W]
        inp = torch.cat([history.reshape(B, 3 * N_CHANNELS, H, W), clim_in, geo_b, doy_map], dim=1)

        dz = model(inp)
        z_next = a * history[:, -1] + dz

        # Straight-through physical clamp: tp >= 0 and tcwv >= 0
        clim_tgt = clim_t[tgt_doy]
        rain_min = -clim_tgt[:, RAIN_CH] / anom_std_t[RAIN_CH]
        tcwv_min = -clim_tgt[:, TCWV_CH] / anom_std_t[TCWV_CH]
        z_hard = z_next.clone()
        z_hard[:, RAIN_CH] = torch.maximum(z_next[:, RAIN_CH], rain_min)
        z_hard[:, TCWV_CH] = torch.maximum(z_next[:, TCWV_CH], tcwv_min)
        z_next = z_next + (z_hard - z_next).detach()

        preds_anom.append(z_next)
        history = torch.cat([history[:, 1:], z_next.unsqueeze(1)], dim=1)
        sin_v, cos_v = next_sin, next_cos

    return preds_anom


class AnomalyRolloutLoss(nn.Module):
    def __init__(self, horizon: int, lead_decay: float = 0.95, var_weight: float = 0.08) -> None:
        super().__init__()
        lw = torch.tensor([lead_decay ** k for k in range(horizon)], dtype=torch.float32)
        self.register_buffer("lead_weights", lw / lw.sum())
        # Channels are already balanced by anom_std; only mild tp nudge
        cw = torch.tensor([1.0, 1.1, 1.0, 1.0, 1.0, 1.0], dtype=torch.float32)
        self.register_buffer("chan_weights", cw / cw.sum())
        self.var_weight = var_weight
        # Two-sided target spatial std ratio for tp: ~0.90 at d1 -> ~0.70 at d8
        vr = torch.tensor([max(0.70, 0.90 - 0.03 * k) for k in range(horizon)], dtype=torch.float32)
        self.register_buffer("target_std_ratio", vr)

    def forward(self, preds: list, targets: torch.Tensor) -> torch.Tensor:
        total = torch.tensor(0.0, device=targets.device)
        horizon = min(len(preds), targets.shape[1])
        for k in range(horizon):
            pk, tk = preds[k], targets[:, k]
            # Smooth Huber with beta=2.0 behaves like MSE on typical anomalies
            err = F.huber_loss(pk, tk, delta=2.0, reduction="none").mean(dim=(0, 2, 3))
            step_loss = (self.chan_weights * err).sum()
            if self.var_weight > 0.0:
                std_p = (pk[:, RAIN_CH].var(dim=(-2, -1), unbiased=False) + 1e-6).sqrt()
                std_t = (tk[:, RAIN_CH].var(dim=(-2, -1), unbiased=False) + 1e-6).sqrt()
                var_loss = ((std_p / (std_t + 0.1)) - self.target_std_ratio[k]).pow(2).mean()
                step_loss = step_loss + self.var_weight * var_loss
            total = total + self.lead_weights[k] * step_loss
        return total


In [ ]:
# ===========================================================================
# GPU-Resident Training Loop with EMA & Early Stopping
# ===========================================================================
torch.manual_seed(SEED)
np.random.seed(SEED)

reader = DatasetReader(NC_PATH)
clim_np, anom_std_np, anom_mean_np = build_smoothed_climatology(reader)
geo_np = build_geo_features(reader.static[0], reader.static[1], reader.lats, reader.lons)

# Preload entire dataset (~170 MB) onto GPU for zero-CPU-overhead batching
clim_t = torch.from_numpy(clim_np).to(device)
anom_std_t = torch.from_numpy(anom_std_np).to(device)
clim_norm_t = ((clim_t - torch.from_numpy(anom_mean_np).to(device).view(1, 6, 1, 1)) / anom_std_t.view(1, 6, 1, 1))
geo_t = torch.from_numpy(geo_np).unsqueeze(0).to(device)

data_phys_t = torch.from_numpy(reader.data).to(device)
doys_t = torch.from_numpy(reader.doys).long().to(device)
data_anom_t = (data_phys_t - clim_t[doys_t]) / anom_std_t.view(1, 6, 1, 1)

doy_angles = 2.0 * math.pi * (doys_t.float() + 1.0) / 365.0
doy_sin_all = torch.sin(doy_angles)
doy_cos_all = torch.cos(doy_angles)

def make_split_starts(split: str, horizon: int) -> torch.Tensor:
    idx = reader.split_indices(split)
    total = 3 + horizon
    starts = [int(idx[i]) for i in range(len(idx) - total + 1) if np.all(np.diff(idx[i:i+total]) == 1)]
    return torch.tensor(starts, dtype=torch.long, device=device)

model = GRASTv3(c_in=C_IN, base_ch=BASE_CH, dropout=DROPOUT).to(device)
ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad_(False)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"GRASTv3 params: {n_params:,} | anom_std: {np.round(anom_std_np, 3).tolist()}")

@torch.no_grad()
def update_ema():
    for ep, mp in zip(ema_model.parameters(), model.parameters()):
        ep.mul_(EMA_DECAY).add_(mp.data, alpha=1.0 - EMA_DECAY)

def sample_batch(starts: torch.Tensor, horizon: int):
    # starts: [B]
    inp_idx = starts[:, None] + torch.arange(3, device=device)[None, :]
    tgt_idx = starts[:, None] + torch.arange(3, 3 + horizon, device=device)[None, :]
    last_idx = starts + 2
    return data_anom_t[inp_idx], data_anom_t[tgt_idx], doy_sin_all[last_idx], doy_cos_all[last_idx]

best_final = {}
out_path = OUT_DIR / "primary_grast_v3.pt"

for phase_h, max_ep, phase_lr in CURRICULUM_PHASES:
    print(f"\n{'='*60}\nPhase: horizon={phase_h}, max_epochs={max_ep}, lr={phase_lr:.2e}\n{'='*60}")
    tr_starts = make_split_starts("train", phase_h)
    va_starts = make_split_starts("val", phase_h)
    optimizer = torch.optim.AdamW(model.parameters(), lr=phase_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_ep, eta_min=max(phase_lr * 0.05, 1e-5))
    loss_fn = AnomalyRolloutLoss(horizon=phase_h).to(device)

    best_val = math.inf
    best_state = None
    wait = 0

    for epoch in range(1, max_ep + 1):
        t0 = time.time()
        model.train()
        perm = tr_starts[torch.randperm(len(tr_starts), device=device)]
        tr_loss, n_tr = 0.0, 0
        for i in range(0, len(perm), BATCH_SIZE):
            b_starts = perm[i : i + BATCH_SIZE]
            x_a, y_a, ds, dc = sample_batch(b_starts, phase_h)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                preds = anomaly_rollout(model, geo_t, clim_t, clim_norm_t, anom_std_t, x_a, ds, dc, phase_h, INPUT_NOISE)
                loss = loss_fn(preds, y_a)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            update_ema()
            tr_loss += loss.item()
            n_tr += 1
        scheduler.step()

        ema_model.eval()
        va_loss, n_va = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(va_starts), BATCH_SIZE):
                x_a, y_a, ds, dc = sample_batch(va_starts[i : i + BATCH_SIZE], phase_h)
                with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
                    preds = anomaly_rollout(ema_model, geo_t, clim_t, clim_norm_t, anom_std_t, x_a, ds, dc, phase_h, 0.0)
                    va_loss += loss_fn(preds, y_a).item()
                n_va += 1

        avg_tr, avg_va = tr_loss / n_tr, va_loss / n_va
        dt = time.time() - t0
        entry = {"phase": f"h{phase_h}", "epoch": epoch, "train_loss": avg_tr, "val_loss": avg_va, "elapsed_s": round(dt, 2)}
        with open(LOG_PATH, "a") as f:
            f.write(json.dumps(entry) + "\n")
        print(f"[h{phase_h}] {epoch:02d}/{max_ep:02d}  train={avg_tr:.4f}  val={avg_va:.4f}  ({dt:.1f}s)")

        if avg_va < best_val:
            best_val = avg_va
            best_state = {k: v.cpu().clone() for k, v in ema_model.state_dict().items()}
            best_final = dict(entry)
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Early stopping h{phase_h} at epoch {epoch} (best val={best_val:.4f})")
                break

    if best_state is not None:
        loaded = {k: v.to(device) for k, v in best_state.items()}
        model.load_state_dict(loaded)
        ema_model.load_state_dict(loaded)

artifact = {
    "model_name": "primary_grast_v3",
    "arch_version": "v3_anomaly",
    "c_in": C_IN,
    "base_ch": BASE_CH,
    "n_params": n_params,
    "horizon": HORIZON,
    "best_metrics": best_final,
    "state_dict": {k: v.cpu() for k, v in ema_model.state_dict().items()},
    "clim": clim_np,
    "anom_std": anom_std_np,
    "anom_mean": anom_mean_np,
}
torch.save(artifact, out_path)
print(f"\nSaved v3 checkpoint to {out_path} ({out_path.stat().st_size / 1e6:.2f} MB)")


In [ ]:
# ===========================================================================
# Built-in Locked Test (2020-2021) Benchmark & Causal Shock Check
# ===========================================================================
@torch.no_grad()
def run_v3_benchmark():
    ema_model.eval()
    w = np.cos(np.deg2rad(reader.lats)).astype("float64")
    w2d = np.broadcast_to(w[:, None], (GRID_H, GRID_W)).copy()
    w2d /= w2d.sum()

    te_starts = make_split_starts("test", 10)
    leads = [1, 4, 8, 10]
    lead_idx = [l - 1 for l in leads]
    mse_acc = np.zeros((3, len(leads), N_CHANNELS), dtype="float64")
    rain10_cm = np.zeros((3, len(leads), 3), dtype="int64")
    rain25_cm = np.zeros((3, len(leads), 3), dtype="int64")
    var_pred = np.zeros((len(leads), N_CHANNELS), dtype="float64")
    var_true = np.zeros((len(leads), N_CHANNELS), dtype="float64")
    p99_p, p99_t = [], []
    n_samp = 0

    for i in range(0, len(te_starts), 64):
        bs = te_starts[i : i + 64]
        B = len(bs)
        x_a, _, ds, dc = sample_batch(bs, 10)
        preds_a = anomaly_rollout(ema_model, geo_t, clim_t, clim_norm_t, anom_std_t, x_a, ds, dc, 10, 0.0)

        tgt_idx = bs[:, None] + torch.arange(3, 13, device=device)[None, :]
        tgt_doys = doys_t[tgt_idx]
        pred_phys = torch.stack([
            preds_a[k] * anom_std_t.view(1, 6, 1, 1) + clim_t[tgt_doys[:, k]]
            for k in range(10)
        ], dim=1).cpu().numpy()
        pred_phys[:, :, RAIN_CH] = np.maximum(pred_phys[:, :, RAIN_CH], 0.0)
        pred_phys[:, :, TCWV_CH] = np.maximum(pred_phys[:, :, TCWV_CH], 0.0)

        true_phys = data_phys_t[tgt_idx].cpu().numpy()
        pers_phys = np.broadcast_to(data_phys_t[bs + 2][:, None].cpu().numpy(), true_phys.shape)
        clim_phys = clim_t[tgt_doys].cpu().numpy()

        for li, k in enumerate(lead_idx):
            tk, pk = true_phys[:, k], pred_phys[:, k]
            var_pred[li] += pk.var(axis=(2, 3)).sum(axis=0)
            var_true[li] += tk.var(axis=(2, 3)).sum(axis=0)
            for mi, cand in enumerate((pred_phys, pers_phys, clim_phys)):
                ck = cand[:, k]
                mse_acc[mi, li] += (((ck - tk) ** 2) * w2d[None, None]).sum(axis=(2, 3)).sum(axis=0)
                for cm, thr in ((rain10_cm, 10.0), (rain25_cm, 25.0)):
                    pr, tr = ck[:, 1] > thr, tk[:, 1] > thr
                    cm[mi, li, 0] += int((pr & tr).sum())
                    cm[mi, li, 1] += int((~pr & tr).sum())
                    cm[mi, li, 2] += int((pr & ~tr).sum())
        p99_p.append(pred_phys[:, 7, 1].ravel())
        p99_t.append(true_phys[:, 7, 1].ravel())
        n_samp += B

    rmse = np.sqrt(mse_acc / n_samp)
    print("\n" + "=" * 84)
    print("GRAST v3 LOCKED TEST SUMMARY (2020-2021)")
    print("=" * 84)
    print("Lead | " + " | ".join(f"{c:>7s}" for c in CHANNELS) + " | MeanRef | CSI>10 | CSI>25")
    print("-" * 84)
    mean_skills = []
    csi_ratios = []
    for li, d in enumerate(leads):
        skills = [1.0 - rmse[0, li, ci] / min(rmse[1, li, ci], rmse[2, li, ci]) for ci in range(6)]
        ms = float(np.mean(skills))
        if d in (1, 4, 8):
            mean_skills.append(ms)
        h10, m10, fa10 = rain10_cm[0, li]
        csi10 = h10 / max(h10 + m10 + fa10, 1)
        h25, m25, fa25 = rain25_cm[0, li]
        csi25 = h25 / max(h25 + m25 + fa25, 1)
        if d in (1, 4, 8):
            rb = max(
                rain10_cm[1, li, 0] / max(rain10_cm[1, li].sum(), 1),
                rain10_cm[2, li, 0] / max(rain10_cm[2, li].sum(), 1),
                1e-4,
            )
            csi_ratios.append(min(csi10 / rb, 1.5) / 1.5)
        vals = " | ".join(f"{s:+7.3f}" for s in skills)
        print(f"d{d:<3d} | {vals} | {ms:+7.3f} | {csi10:6.3f} | {csi25:6.3f}")

    d8_vr = {c: round(float(var_pred[2, ci] / max(var_true[2, ci], 1e-12)), 4) for ci, c in enumerate(CHANNELS)}
    print("\nDay-8 Spatial Variance Ratio [0.40, 2.50]:", d8_vr)
    print("Day-8 Rain P99 (mm/day):", {"pred": round(float(np.percentile(np.concatenate(p99_p), 99)), 2),
                                         "true": round(float(np.percentile(np.concatenate(p99_t), 99)), 2)})
    pts_err = float(np.clip(np.mean(mean_skills) / 0.30, 0.0, 1.0) * 30.0)
    pts_rain = float(np.mean(csi_ratios) * 15.0)
    pts_blur = float(sum(1 for v in d8_vr.values() if 0.40 <= v <= 2.50) / 6.0 * 10.0)
    print(f"Proxy Skill+Rain+AntiBlur (out of 55, excl. shocks): {pts_err + pts_rain + pts_blur:.2f} "
          f"(err={pts_err:.2f}/30, rain={pts_rain:.2f}/15, blur={pts_blur:.2f}/10)")

run_v3_benchmark()
